# 01 Explore MART

## Cosa fa / Cosa NON fa

- apre una tabella mart e ne mostra una lettura iniziale
- prova a scegliere automaticamente una colonna anno e una metrica numerica
- se non trova colonne adatte, salta le query dipendenti e mostra istruzioni

In [ ]:
from pathlib import Path
import duckdb

ROOT = Path('.').resolve()
TABLE_NAME = 'project_summary'
MART_GLOBS = [
    (ROOT / '..' / 'data' / 'mart').resolve(),
    (ROOT / '..' / '_runs').resolve(),
]

def first_match(table_name):
    for base in MART_GLOBS:
        if not base.exists():
            continue
        for path in sorted(base.glob(f'**/*{table_name}*.parquet')):
            return path
    return None

mart_path = first_match(TABLE_NAME)
mart_path

Descrizione breve:

- questo notebook serve a capire cosa c'è nella tabella finale
- aggiorna `TABLE_NAME` se il dataset usa un altro mart principale

In [ ]:
con = duckdb.connect()

def read_schema(path):
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    schema.columns = [str(col).lower() for col in schema.columns]
    return schema

def choose_columns(schema):
    name_col = 'column_name' if 'column_name' in schema.columns else schema.columns[0]
    type_col = 'column_type' if 'column_type' in schema.columns else schema.columns[1]
    rows = [
        {'name': str(row[name_col]), 'type': str(row[type_col]).upper()}
        for _, row in schema.iterrows()
    ]
    year_col = next((r['name'] for r in rows if r['name'].lower() == 'year' or 'anno' in r['name'].lower()), None)
    numeric_rows = [r for r in rows if any(token in r['type'] for token in ['INT', 'DECIMAL', 'DOUBLE', 'FLOAT', 'REAL', 'BIGINT'])]
    metric_col = next((r['name'] for r in numeric_rows if any(token in r['name'].lower() for token in ['value', 'tot', 'importo', 'ammontare', 'pct', 'percent'])), None)
    if metric_col is None and numeric_rows:
        metric_col = numeric_rows[0]['name']
    return year_col, metric_col

if mart_path:
    schema_df = read_schema(str(mart_path))
    YEAR_COL, METRIC_COL = choose_columns(schema_df)
    preview = con.execute(f"SELECT * FROM read_parquet('{mart_path}') LIMIT 20").df()
    display(schema_df)
    display(preview)
    print({'YEAR_COL': YEAR_COL, 'METRIC_COL': METRIC_COL})
else:
    print('No mart parquet found. Run the pipeline first or update TABLE_NAME.')

In [ ]:
if mart_path and YEAR_COL and METRIC_COL:
    by_year = con.execute(
        f"SELECT {YEAR_COL} AS year_like, COUNT(*) AS rows, SUM({METRIC_COL}) AS metric_total FROM read_parquet('{mart_path}') GROUP BY 1 ORDER BY 1"
    ).df()
    display(by_year)
else:
    print('No year-like column or metric column detected. Update TABLE_NAME or inspect schema_df manually.')

In [ ]:
if mart_path and METRIC_COL:
    top_rows = con.execute(
        f"SELECT * FROM read_parquet('{mart_path}') ORDER BY {METRIC_COL} DESC NULLS LAST LIMIT 10"
    ).df()
    bottom_rows = con.execute(
        f"SELECT * FROM read_parquet('{mart_path}') ORDER BY {METRIC_COL} ASC NULLS LAST LIMIT 10"
    ).df()
    display(top_rows)
    display(bottom_rows)
else:
    print('No metric column detected. Update the helper or inspect schema_df manually.')